In [ ]:
from pathlib import Path
import os
import sys

PROJECT_DIR = Path('/workspace/exaone35')
if not PROJECT_DIR.exists():
    PROJECT_DIR = Path.cwd()

os.chdir(PROJECT_DIR)

MODEL_ID = 'LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct'
SYSTEM_PROMPT = 'You are EXAONE model from LG AI Research, a helpful assistant.'

os.environ['HF_HOME'] = '/workspace/.cache/huggingface'
os.environ['MODEL_ID'] = MODEL_ID
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['DISABLE_DOTENV'] = '1'
os.environ.pop('HF_TOKEN', None)

print('project_dir:', PROJECT_DIR)
print('python:', sys.executable)
print('hf_home:', os.environ['HF_HOME'])
print('model_id:', MODEL_ID)
print('disable_dotenv:', os.environ['DISABLE_DOTENV'])

In [ ]:
!nvidia-smi
!python3 --version || true

BF16 LoFA 기본으로 사용

In [ ]:
import importlib.metadata as metadata
import torch

packages = ['torch', 'transformers', 'accelerate', 'datasets', 'peft', 'trl', 'bitsandbytes', 'huggingface_hub']
for name in packages:
    try:
        print(f'{name}:', metadata.version(name))
    except metadata.PackageNotFoundError:
        print(f'{name}: NOT_INSTALLED')

print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
    print('torch cuda:', torch.version.cuda)
    print('bf16 supported:', torch.cuda.is_bf16_supported())
    free, total = torch.cuda.mem_get_info(0)
    print('gpu memory free GB:', round(free / 1024**3, 2))
    print('gpu memory total GB:', round(total / 1024**3, 2))

In [ ]:
from huggingface_hub import notebook_login

os.environ.pop('HF_TOKEN', None)
notebook_login(skip_if_logged_in=False)

In [ ]:
from huggingface_hub import get_token, whoami

HF_TOKEN = get_token()
if not HF_TOKEN:
    raise RuntimeError('Hugging Face 토큰이 저장되지 않았습니다. 로그인 셀을 다시 실행하세요.')

info = whoami(token=HF_TOKEN)
print('logged in user:', info.get('name'))
print('token prefix:', HF_TOKEN[:6] + '***')

In [ ]:
import json
from pathlib import Path

TRAIN_FILE = Path('data/train.jsonl')
VALID_FILE = Path('data/valid.jsonl')
VALID_ROLES = {'system', 'user', 'assistant'}

def validate_jsonl(path: Path) -> int:
    count = 0
    with path.open(encoding='utf-8') as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            count += 1
            row = json.loads(line)
            messages = row.get('messages')
            if not isinstance(messages, list) or not messages:
                raise ValueError(f'{path}:{line_no}: messages must be a non-empty list')
            if messages[-1].get('role') != 'assistant':
                raise ValueError(f'{path}:{line_no}: last message must be assistant')
            roles = [m.get('role') for m in messages]
            if 'user' not in roles or 'assistant' not in roles:
                raise ValueError(f'{path}:{line_no}: user and assistant messages are required')
            for idx, message in enumerate(messages):
                if message.get('role') not in VALID_ROLES:
                    raise ValueError(f'{path}:{line_no}: invalid role at message {idx}')
                if not isinstance(message.get('content'), str) or not message['content'].strip():
                    raise ValueError(f'{path}:{line_no}: empty content at message {idx}')
    return count

print('train rows:', validate_jsonl(TRAIN_FILE))
print('valid rows:', validate_jsonl(VALID_FILE))
print('validation: OK')

In [ ]:
with TRAIN_FILE.open(encoding='utf-8') as f:
    first_sample = json.loads(next(f))

first_sample

In [ ]:
from datasets import load_dataset

raw_dataset = load_dataset('json', data_files={
    'train': str(TRAIN_FILE),
    'validation': str(VALID_FILE),
})

raw_dataset

In [ ]:
from transformers import AutoTokenizer

DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
print('selected dtype:', DTYPE)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    token=HF_TOKEN,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

print('vocab size:', len(tokenizer))
print('pad token:', tokenizer.pad_token, tokenizer.pad_token_id)
print('eos token:', tokenizer.eos_token, tokenizer.eos_token_id)

In [ ]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    device_map='auto',
    trust_remote_code=True,
    low_cpu_mem_usage=True,
    token=HF_TOKEN,
)
model.eval()

first_device = next(model.parameters()).device
print('loaded:', True)
print('first parameter device:', first_device)
if torch.cuda.is_available():
    print('memory allocated GB:', round(torch.cuda.memory_allocated() / 1024**3, 3))
    print('memory reserved GB:', round(torch.cuda.memory_reserved() / 1024**3, 3))

In [ ]:
prompt = '한국어 형태소 분석이 왜 중요한지 짧게 설명해줘.'
messages = [
    {'role': 'system', 'content': SYSTEM_PROMPT},
    {'role': 'user', 'content': prompt},
]

# 최신 Transformers에서는 apply_chat_template가 BatchEncoding을 반환할 수 있습니다.
# 그래서 Tensor 하나를 generate에 넘기지 않고, input_ids/attention_mask를 dict로 풀어 전달합니다.
encoded = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors='pt',
    return_dict=True,
).to(first_device)

inputs = dict(encoded)
prompt_length = inputs['input_ids'].shape[-1]

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

new_tokens = output_ids[0][prompt_length:]
response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
print(response)

학습전 gpu메모리 정리

In [ ]:
del encoded
del inputs
del output_ids
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

if torch.cuda.is_available():
    print('memory allocated GB:', round(torch.cuda.memory_allocated() / 1024**3, 3))
    print('memory reserved GB:', round(torch.cuda.memory_reserved() / 1024**3, 3))